# aw_09_b6 — Stage B6: PlayWorld GRPO from the Phase-2 champion (Track B, §5.1)

**Parent = B4v2** (`20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2`,
sha256:d4fcacdd...; §6 Phase-2 champion — B5 DPO was ~null vs B4v2, so GRPO also
starts from the SFT champion). This gives the clean triangle from ONE parent:
B4v2 (SFT) vs B5 (offline DPO) vs B6 (online GRPO), all on the frozen suites.

Rewards: `verifier_reward_function(default_playworld_verifier)` — the SAME
verifier as eval, applied online to rollouts (episode-level, no reward model).
Prompts: canonical frozen `train/v1/playworld_prompts.jsonl` (v1.3 rule).

Cell order: fetch parent → `a_b6_data` → `b_b6_train` → `c_b6_eval` →
`x09j_run_audit` → `f_b6_analysis` (B6 vs B4v2, B6 vs B5 = headline).


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title fetch parent — B4v2 champion adapter + lineage sha
B4V2_RUN_ID = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_RUN_ID}
b4v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b4v2_sha = json.load(open(f"runs/{B4V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b4v2_dir, b4v2_sha)


In [ ]:
# @title a_b6_data — frozen prompts + suites (sha-pinned, v1.3 rule)
!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld --path train/v1/playworld_prompts.jsonl \
  --output data/train/playworld_prompts.jsonl --force \
  --expected-sha256 2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_fingerprint MUST equal sha256:3cdcbc30... — abort otherwise


In [ ]:
# @title b_b6_train — GRPO from the B4v2 parent (online verifier reward)
# Watch reward/mean in the logs: it should start near B4v2's train-distribution
# pass level and climb. If reward flatlines at 0 or 1, stop and report.
!python scripts/run_experiment.py \
  --config configs/experiments/b6_playworld_grpo.yaml \
  --parent-adapter-dir {b4v2_dir} \
  --override lineage.parent_run_id={B4V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4v2_sha} \
  --override data.source.local_path=data/train/playworld_prompts.jsonl \
  --hf-sync-repo m97j/aw-runs-b6


In [ ]:
# @title c_b6_eval — B6 adapter on the frozen suites (canonical profile)
B6_RUN_ID = ""  # <- from b_b6_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6_RUN_ID}
b6_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b6_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b6


In [ ]:
# @title x09j_run_audit — termination regression check on the B6 eval (CPU)
B6_EVAL = ""  # <- eval run id from c_b6_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B6_EVAL} --out runs/x09_run_audit_b6.json


In [ ]:
# @title f_b6_analysis — B6 vs B4v2 (RL gain) and B6 vs B5 (online vs offline)
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"
B5_EVAL = "20260814-124224--eval-playworld--s42--f77cd8"

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b5 --run-id {B5_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B6_EVAL} --label-a b6-grpo --run-b runs/{B4V2_EVAL} --label-b b4v2-sft \
  --output runs/{B6_EVAL}/analysis_b6_vs_b4v2.json --hf-sync-repo m97j/aw-runs-b6

!python scripts/run_analysis.py \
  --run-a runs/{B6_EVAL} --label-a b6-grpo --run-b runs/{B5_EVAL} --label-b b5-dpo \
  --output runs/{B6_EVAL}/analysis_b6_vs_b5.json --hf-sync-repo m97j/aw-runs-b6


## Stage checklist
- [ ] a_b6_data: prompts sha 2e7d0260 verified; suites freeze 3cdcbc30 verified
- [ ] b_b6_train: reward/mean trajectory recorded (start≈parent level, climbing);
      lineage verified (parent sha d4fcacdd)
- [ ] c_b6_eval per-suite pass_rate; x09j truncation/runaway ≈ 0
- [ ] f_b6_analysis: B6 vs B4v2 and **B6 vs B5** recorded
- [ ] §6 final Track-B champion (B4v2/B5/B6) → seeds (aw_11) + ablations (aw_10) + report
